# Intent Classification — Linear SVM

This notebook trains and validates the final intent classifier used by the banking support LangGraph workflow.

### Final intent classes
- Fraud/Unauthorized
- Loan
- KYC
- Account Access

The model uses TF-IDF word + character features with Linear SVM. The trained pipeline is saved as `models/intent_model.pkl` and loaded by `src/ml/intent.py` at runtime.

## 1. Setup

In [1]:
from pathlib import Path

import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path.cwd().resolve().parent
TICKETS_PATH = PROJECT_ROOT / "data" / "support_tickets.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "intent_model.pkl"

print("Project root:", PROJECT_ROOT)
print("Tickets file exists:", TICKETS_PATH.exists())

Project root: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System
Tickets file exists: True


## 2. Build the Intent Dataset

The support-ticket data contains repeated customer-query templates. For intent classification, keep one row per unique `query_text` so duplicate copies of the same query do not inflate the training set.

In [2]:
tickets = pd.read_csv(TICKETS_PATH)

intent_data = (
    tickets[["query_text", "category"]]
    .drop_duplicates(subset="query_text")
    .reset_index(drop=True)
)

print("Intent dataset shape:", intent_data.shape)
print("\nClass distribution:")
display(intent_data["category"].value_counts())

print("\nConflicting query/category mappings:")
query_category_check = (
    intent_data.groupby("query_text")["category"].nunique()
)
display(query_category_check[query_category_check > 1])

display(intent_data.head(10))

Intent dataset shape: (66, 2)

Class distribution:


category
Fraud/Unauthorized    32
Loan                  14
KYC                   10
Account Access        10
Name: count, dtype: int64


Conflicting query/category mappings:


Series([], Name: category, dtype: int64)

,query_text,category
0,I see a transaction of ₹999 I didn't make on 2...,Fraud/Unauthorized
1,I see a transaction of ₹500 I didn't make on 1...,Fraud/Unauthorized
2,"I see a transaction of ₹25,000 I didn't make o...",Fraud/Unauthorized
3,I see a transaction of ₹999 I didn't make on 1...,Fraud/Unauthorized
4,"I see a transaction of ₹25,000 I didn't make o...",Fraud/Unauthorized
5,"There's an ATM withdrawal of ₹5,000 from Banga...",Fraud/Unauthorized
6,"There's an ATM withdrawal of ₹7,500 from Hyder...",Fraud/Unauthorized
7,"There's an ATM withdrawal of ₹2,000 from Hyder...",Fraud/Unauthorized
8,"There's an ATM withdrawal of ₹15,000 from Hyde...",Fraud/Unauthorized
9,There's an ATM withdrawal of ₹500 from Hyderab...,Fraud/Unauthorized


## 3. Train / Test Split

In [3]:
X = intent_data["query_text"]
y = intent_data["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTraining class distribution:")
display(y_train.value_counts())

print("Test class distribution:")
display(y_test.value_counts())

Training samples: 52
Test samples: 14

Training class distribution:


category
Fraud/Unauthorized    25
Loan                  11
KYC                    8
Account Access         8
Name: count, dtype: int64

Test class distribution:


category
Fraud/Unauthorized    7
Loan                  3
KYC                   2
Account Access        2
Name: count, dtype: int64

## 4. Baseline — Word TF-IDF + Linear SVM

In [4]:
word_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            sublinear_tf=True,
        ),
    ),
    (
        "classifier",
        LinearSVC(C=1.0),
    ),
])

word_model.fit(X_train, y_train)
y_pred_word = word_model.predict(X_test)

word_accuracy = accuracy_score(y_test, y_pred_word)

print("Baseline accuracy:", round(word_accuracy, 4))

Baseline accuracy: 0.5714


## 5. Final Model — Word + Character TF-IDF + Linear SVM

Character n-grams help with short banking phrases, spelling variations, and token-level differences while word n-grams capture the main intent vocabulary.

In [5]:
intent_model = Pipeline([
    (
        "features",
        FeatureUnion([
            (
                "word_tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    sublinear_tf=True,
                ),
            ),
            (
                "char_tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    sublinear_tf=True,
                ),
            ),
        ]),
    ),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            class_weight="balanced",
        ),
    ),
])

intent_model.fit(X_train, y_train)
y_pred = intent_model.predict(X_test)

final_accuracy = accuracy_score(y_test, y_pred)

print("Final accuracy:", round(final_accuracy, 4))
print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0,
    )
)

Final accuracy: 0.6429

Classification report:
                    precision    recall  f1-score   support

    Account Access       0.33      1.00      0.50         2
Fraud/Unauthorized       0.80      0.57      0.67         7
               KYC       1.00      0.50      0.67         2
              Loan       1.00      0.67      0.80         3

          accuracy                           0.64        14
         macro avg       0.78      0.68      0.66        14
      weighted avg       0.80      0.64      0.67        14



In [6]:
comparison = pd.DataFrame(
    {
        "Model": [
            "Word TF-IDF + Linear SVM",
            "Word + Char TF-IDF + Linear SVM",
        ],
        "Accuracy": [
            word_accuracy,
            final_accuracy,
        ],
    }
).sort_values("Accuracy", ascending=False)

display(comparison.round(4))

,Model,Accuracy
1,Word + Char TF-IDF + Linear SVM,0.6429
0,Word TF-IDF + Linear SVM,0.5714


## 6. Error Analysis

In [7]:
prediction_df = pd.DataFrame(
    {
        "query_text": X_test.values,
        "actual": y_test.values,
        "predicted": y_pred,
    }
)

misclassified = prediction_df[
    prediction_df["actual"] != prediction_df["predicted"]
].copy()

print("Misclassified queries:", len(misclassified))
display(misclassified)

Misclassified queries: 5


,query_text,actual,predicted
3,Someone used my credit card online without my ...,Fraud/Unauthorized,Account Access
4,My UPI ID was used to transfer money without m...,Fraud/Unauthorized,Account Access
6,My EMI was deducted twice this month,Loan,Fraud/Unauthorized
7,My PAN verification is failing on net banking,KYC,Account Access
10,Multiple small transactions from my account I ...,Fraud/Unauthorized,Account Access


In [8]:
labels = intent_model.classes_

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels,
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual: {label}" for label in labels],
    columns=[f"Predicted: {label}" for label in labels],
)

display(cm_df)

,Predicted: Account Access,Predicted: Fraud/Unauthorized,Predicted: KYC,Predicted: Loan
Actual: Account Access,2,0,0,0
Actual: Fraud/Unauthorized,3,4,0,0
Actual: KYC,1,0,1,0
Actual: Loan,0,1,0,2


## 7. 5-Fold Cross-Validation

Cross-validation is reported as an additional robustness check. The holdout test set remains the main final evaluation used for the model artifact.

In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_scores = cross_val_score(
    intent_model,
    X,
    y,
    cv=cv,
    scoring="accuracy",
)

print("5-Fold CV scores:", cv_scores.round(3))
print("Mean CV accuracy:", round(cv_scores.mean(), 4))
print("CV std:", round(cv_scores.std(), 4))

5-Fold CV scores: [0.786 0.846 0.769 0.692 0.846]
Mean CV accuracy: 0.7879
CV std: 0.0571


## 8. Save the Final Model

In [10]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(
    intent_model,
    MODEL_PATH,
)

print("Intent model saved:")
print(MODEL_PATH)

Intent model saved:
E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System\models\intent_model.pkl


## 9. Reload and Smoke Test

In [11]:
loaded_model = joblib.load(MODEL_PATH)

test_queries = [
    "I cannot complete my Aadhaar verification",
    "I want to know the documents needed for a home loan",
    "Someone used my credit card without my permission",
    "I am unable to access my net banking account",
]

smoke_test = pd.DataFrame(
    {
        "query": test_queries,
        "predicted_intent": loaded_model.predict(test_queries),
    }
)

display(smoke_test)

,query,predicted_intent
0,I cannot complete my Aadhaar verification,KYC
1,I want to know the documents needed for a home...,Loan
2,Someone used my credit card without my permission,Account Access
3,I am unable to access my net banking account,Account Access


## 10. Runtime Design Note

The notebook saves the base Linear SVM pipeline. Runtime safety handling is implemented separately in `src/ml/intent.py`, where `predict_intent_safe()` combines the model prediction with fraud-specific safety rules.

This keeps the trained model simple while allowing the production application to apply deterministic safety overrides.

## Final Result

The exported artifact is:

`models/intent_model.pkl`

This is the model loaded by the LangGraph application.